# Atualizando definições de tools dos seus MCP servers no AgentCore Gateway

## Visão Geral

O mecanismo de sincronização no AgentCore Gateway garante definições precisas de tools a partir de MCP server targets. Ele gerencia a consistência de schema, otimiza o desempenho e mantém a integridade dos dados por meio de métodos de sincronização explícita e implícita.

### Sincronização Explícita

A API SynchronizeGateway é uma operação assíncrona que permite a sincronização sob demanda de tools a partir de MCP server targets. Esta API fornece aos clientes controle explícito sobre quando atualizar suas definições de tools, sendo particularmente útil após fazer alterações nas configurações de tools do seu MCP server. Para targets configurados com autenticação OAuth, a API primeiro valida as credenciais por meio do serviço AgentCore Identity antes de prosseguir com a comunicação com o MCP server. Se a validação falhar, a operação de sincronização falha com detalhes de erro apropriados, transicionando o target para um estado FAILED. Para targets configurados sem autenticação, a API prossegue diretamente para a sincronização de tools.

O fluxo de trabalho de processamento de tools começa com o estabelecimento de uma sessão com o MCP server. A API então recupera e processa tools em lotes otimizados, adicionando prefixos específicos do target para evitar colisões de nomes com tools de outros targets. A API impõe um limite de 10.000 tools por target e garante que as definições de tools sejam normalizadas, preservando metadados essenciais das definições originais do MCP server.

Neste tutorial, vamos atualizar o MCP Server criado em '01-Gateway-MCP-Target' adicionando tools adicionais e então invocar a API SynchronizeGateway para sincronizar explicitamente as definições de tools mais recentes.

![Diagram](images/mcp-server-target-explicit-sync.png)

### Sincronização Implícita

O AgentCore Gateway realiza uma sincronização implícita durante as operações CreateGatewayTarget e UpdateGatewayTarget que difere da API explícita SynchronizeGateway. Esta sincronização integrada garante que MCP targets recém-criados ou atualizados sejam imediatamente utilizáveis, mantendo a consistência dos dados. Esta sincronização implícita garante que MCP targets sejam sempre criados ou atualizados com definições de tools válidas e atuais, mantendo a garantia do Gateway de que qualquer target no estado READY esteja imediatamente utilizável. 

Neste tutorial, vamos criar um novo MCP Server e chamar a operação UpdateGatewayTarget para atualizar o novo MCP target que sincroniza implicitamente as definições de tools do novo MCP Server.

![Diagram](images/mcp-server-target-implicit-sync.png)

### Detalhes do Tutorial


| Informação           | Detalhes                                                                         |
|:---------------------|:---------------------------------------------------------------------------------|
| Tipo do tutorial     | Interativo                                                                       |
| Componentes AgentCore| AgentCore Gateway, AgentCore Identity, AgentCore Runtime                         |
| Framework de Agentes | Strands Agents                                                                   |
| Tipo de Gateway Target| MCP server                                                                      |
| Agente               | Strands                                                                          |
| Inbound Auth IdP| Amazon Cognito, mas pode usar outros                                            |
| Outbound Auth        | Amazon Cognito, mas pode usar outros                                             |
| Modelo LLM           | Anthropic Claude Sonnet 4                                                        |
| Componentes do tutorial| Criando AgentCore Gateway com MCP Target e sincronizando as tools               |
| Vertical do tutorial | Cross-vertical                                                                   |
| Complexidade do exemplo| Fácil                                                                           |
| SDK utilizado        | boto3                                                                            |

## Arquitetura do Tutorial

Neste tutorial, vamos descrever como acionar tanto a sincronização explícita quanto a implícita de definições de tools a partir de MCP server targets dentro do AgentCore Gateway

## Pré-requisitos

Para executar este tutorial você precisará de:
* Jupyter notebook (kernel Python)
* uv
* Credenciais AWS
* Amazon Cognito

In [ ]:
# Install from the requirements file or pyproject.toml file in current directory
!pip install --force-reinstall -U -r requirements.txt --quiet

In [ ]:
# Set AWS credentials if not using Amazon SageMaker notebook
import os
# os.environ['AWS_ACCESS_KEY_ID'] = '' # Set the access key
# os.environ['AWS_SECRET_ACCESS_KEY'] = '' # Set the secret key
os.environ['AWS_DEFAULT_REGION'] = os.environ.get('AWS_REGION', 'us-east-1')

In [ ]:
# Import utils
import os
import sys

# Get the directory of the current script
if '__file__' in globals():
    current_dir = os.path.dirname(os.path.abspath("."))
else:
    current_dir = os.getcwd()  # Fallback if __file__ is not defined (e.g., Jupyter)

# Navigate to the directory containing utils.py (one level up)
utils_dir = os.path.abspath(os.path.join(current_dir, '..'))

# Add to sys.path
sys.path.insert(0, utils_dir)

# Now you can import utils
import utils

# Setup logging 
import logging

# Configure logging for notebook environment
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
    handlers=[logging.StreamHandler()]
)

# Set specific logger levels
logging.getLogger("gateway").setLevel(logging.INFO)

## Step 2: Create AgentCore Gateway

## Recuperar as Informações do Cognito User Pool para inbound auth no Gateway

In [ ]:
# Creating Cognito User Pool 
import os
import boto3

REGION = os.environ['AWS_DEFAULT_REGION']
USER_POOL_NAME = "sample-agentcore-gateway-pool"
RESOURCE_SERVER_ID = "sample-agentcore-gateway-id"
RESOURCE_SERVER_NAME = "sample-agentcore-gateway-name"
CLIENT_NAME = "sample-agentcore-gateway-client"
SCOPES = [
    {"ScopeName": "invoke",  # Just 'invoke', will be formatted as resource_server_id/invoke
    "ScopeDescription": "Scope for invoking the agentcore gateway"},
]

scope_names = [f"{RESOURCE_SERVER_ID}/{scope['ScopeName']}" for scope in SCOPES]
scopeString = " ".join(scope_names)


cognito = boto3.client("cognito-idp", region_name=REGION)

print("Creating or retrieving Cognito resources...")
gw_user_pool_id = utils.get_or_create_user_pool(cognito, USER_POOL_NAME)
print(f"User Pool ID: {gw_user_pool_id}")

utils.get_or_create_resource_server(cognito, gw_user_pool_id, RESOURCE_SERVER_ID, RESOURCE_SERVER_NAME, SCOPES)
print("Resource server ensured.")

gw_client_id, gw_client_secret = utils.get_or_create_m2m_client(cognito, gw_user_pool_id, CLIENT_NAME, RESOURCE_SERVER_ID, scope_names)

print(f"Client ID: {gw_client_id}")

# Get discovery URL
gw_cognito_discovery_url = f'https://cognito-idp.{REGION}.amazonaws.com/{gw_user_pool_id}/.well-known/openid-configuration'
print(gw_cognito_discovery_url)

## Recuperar as Informações do Cognito User Pool para inbound auth no Runtime

In [ ]:
agentcore_gateway_iam_role = utils.create_agentcore_gateway_role("sample-mcpgateway")
print("AgentCore Gateway role ARN:", agentcore_gateway_iam_role["Role"]["Arn"])

### Step 2.3: Create AgentCore Gateway

In [ ]:
gateway_client = boto3.client("bedrock-agentcore-control", region_name=REGION)

auth_config = {
    "customJWTAuthorizer": {
        "allowedClients": [gw_client_id],
        "discoveryUrl": gw_cognito_discovery_url,
    }
}

# IMPORTANT: searchType="NONE" — Step 7's DYNAMIC target requires this.
# Per gateway-target-MCPservers.html: "Dynamic MCP targets
# (listingMode=DYNAMIC) are not supported on gateways with semantic search
# enabled." Notebook 01 uses searchType="SEMANTIC" because it doesn't create
# DYNAMIC targets.
create_response = gateway_client.create_gateway(
    name="ac-gateway-mcp-server",
    roleArn=agentcore_gateway_iam_role["Role"]["Arn"],
    protocolType="MCP",
    protocolConfiguration={"mcp": {"supportedVersions": ["2025-11-25"]}},
    authorizerType="CUSTOM_JWT",
    authorizerConfiguration=auth_config,
    description="AgentCore Gateway with MCP Server target (sync demos)",
)
gatewayID = create_response["gatewayId"]
gatewayURL = create_response["gatewayUrl"]
print(f"Gateway ID:  {gatewayID}")
print(f"Gateway URL: {gatewayURL}")

## Step 3: Deploy the initial MCP server on AgentCore Runtime

## Recuperar as Informações do Cognito User Pool para inbound auth no Runtime

In [ ]:
# Creating Cognito User Pool
import os
import boto3

REGION = os.environ['AWS_DEFAULT_REGION']
USER_POOL_NAME = "sample-agentcore-runtime-pool"
RESOURCE_SERVER_ID = "sample-agentcore-runtime-id"
RESOURCE_SERVER_NAME = "sample-agentcore-runtime-name"
CLIENT_NAME = "sample-agentcore-runtime-client"
SCOPES = [
    {"ScopeName": "invoke",  # Just 'invoke', will be formatted as resource_server_id/invoke
    "ScopeDescription": "Scope for invoking the agentcore gateway"},
]

scope_names = [f"{RESOURCE_SERVER_ID}/{scope['ScopeName']}" for scope in SCOPES]
runtimeScopeString = " ".join(scope_names)


cognito = boto3.client("cognito-idp", region_name=REGION)

print("Creating or retrieving Cognito resources...")
runtime_user_pool_id = utils.get_or_create_user_pool(cognito, USER_POOL_NAME)
print(f"User Pool ID: {runtime_user_pool_id}")

utils.get_or_create_resource_server(cognito, runtime_user_pool_id, RESOURCE_SERVER_ID, RESOURCE_SERVER_NAME, SCOPES)
print("Resource server ensured.")

runtime_client_id, runtime_client_secret = utils.get_or_create_m2m_client(cognito, runtime_user_pool_id, CLIENT_NAME, RESOURCE_SERVER_ID, scope_names)

print(f"Client ID: {runtime_client_id}")

# Get discovery URL
runtime_cognito_discovery_url = f'https://cognito-idp.{REGION}.amazonaws.com/{runtime_user_pool_id}/.well-known/openid-configuration'
print(runtime_cognito_discovery_url)

### Step 3.2: Write the initial MCP server (`getOrder` + `updateOrder`)

The sync demos in Steps 5 and 6 work by *adding* tools to this server and watching when the gateway notices. Start small: just two tools.

In [ ]:
%%writefile mcp_server.py
from mcp.server.fastmcp import FastMCP

mcp = FastMCP(host="0.0.0.0", stateless_http=True)


@mcp.tool()
def getOrder() -> int:
    """Get an order."""
    return 123


@mcp.tool()
def updateOrder(orderId: int) -> int:
    """Update an existing order."""
    return 456


if __name__ == "__main__":
    mcp.run(transport="streamable-http")

### Step 3.3: Configure and launch on AgentCore Runtime

`deploy_mcp_server` (from `runtime_deploy.py`) wraps configure + launch + URL derivation into one call.

In [ ]:
from runtime_deploy import deploy_mcp_server

deployed = deploy_mcp_server(
    entrypoint="mcp_server.py",
    agent_name="ac_gateway_sync",
    region=REGION,
    runtime_client_id=runtime_client_id,
    runtime_discovery_url=runtime_cognito_discovery_url,
)
agentcore_runtime = deployed.runtime
mcp_arn = deployed.agent_arn
mcp_id = deployed.agent_id
mcp_url = deployed.agent_url

## Step 4: Wire the MCP Server in as a Gateway Target

### Verificar as definições de tools atuais

Vamos invocar um agente strands para listar as MCP tools usando o Bedrock AgentCore Gateway. Isso retornará a lista antiga de tools (sem 'cancelOrder'), porque ainda não pedimos ao Gateway para sincronizar.

In [ ]:
identity_client = boto3.client("bedrock-agentcore-control", region_name=REGION)

cognito_provider = identity_client.create_oauth2_credential_provider(
    name="ac-gateway-mcp-server-identity",
    credentialProviderVendor="CustomOauth2",
    oauth2ProviderConfigInput={
        "customOauth2ProviderConfig": {
            "oauthDiscovery": {"discoveryUrl": runtime_cognito_discovery_url},
            "clientId": runtime_client_id,
            "clientSecret": runtime_client_secret,
        }
    },
)
cognito_provider_arn = cognito_provider["credentialProviderArn"]
print(cognito_provider_arn)

### Step 4.2: Create the Gateway Target

In [ ]:
create_gateway_target_response = gateway_client.create_gateway_target(
    name="mcp-server-target",
    gatewayIdentifier=gatewayID,
    targetConfiguration={"mcp": {"mcpServer": {"endpoint": mcp_url}}},
    credentialProviderConfigurations=[
        {
            "credentialProviderType": "OAUTH",
            "credentialProvider": {
                "oauthCredentialProvider": {
                    "providerArn": cognito_provider_arn,
                    "scopes": [runtimeScopeString],
                }
            },
        },
    ],
)
gatewayTargetID = create_gateway_target_response["targetId"]
print(f"Created target: {gatewayTargetID}")

### Step 4.3: Verify the Gateway Target is READY

In [ ]:
list_targets_response = gateway_client.list_gateway_targets(gatewayIdentifier=gatewayID)
print(list_targets_response)

### Step 4.4: Get an inbound access token

In [ ]:
token_response = utils.get_token(
    gw_user_pool_id, gw_client_id, gw_client_secret, scopeString, REGION
)
token = token_response["access_token"]
print("Token (truncated):", token[:60], "...")

### Step 4.5: Set up the `GatewayMCPClient` helper

`gateway_mcp_client.GatewayMCPClient` (defined alongside the notebook) wraps the bearer-token + `MCP-Protocol-Version` + JSON-RPC plumbing so the demo cells can call `mcp.list_tools()` etc. as one-liners. Created once here, reused throughout the rest of the workshop.

In [ ]:
import json

from gateway_mcp_client import GatewayMCPClient


def _get_inbound_token() -> str:
    return utils.get_token(
        gw_user_pool_id, gw_client_id, gw_client_secret, scopeString, REGION
    )["access_token"]


mcp = GatewayMCPClient(gatewayURL, _get_inbound_token)
print("GatewayMCPClient ready.")

## Step 5: Explicit synchronization with `SynchronizeGatewayTargets`

Adicione uma nova tool 'cancelOrder' ao MCP Server existente que você criou no tutorial '01-mcp-server-target'.

In [ ]:
%%writefile mcp_server_updated.py
from mcp.server.fastmcp import FastMCP

mcp = FastMCP(host="0.0.0.0", stateless_http=True)

@mcp.tool()
def getOrder() -> int:
    """Get an order"""
    return 123

@mcp.tool()
def updateOrder(orderId: int) -> int:
    """Update existing order"""
    return 456

@mcp.tool()
def cancelOrder(orderId: int) -> int:
    """cancel existing order"""
    return 789

if __name__ == "__main__":
    mcp.run(transport="streamable-http")

Em seguida, vamos configurar o AgentCore Runtime para implantar o MCP server atualizado com a nova tool 'cancelOrder'.

In [ ]:
from bedrock_agentcore_starter_toolkit import Runtime
from boto3.session import Session

boto_session = Session()
region = boto_session.region_name
print(f"Using AWS region: {region}")

required_files = ['mcp_server_updated.py', 'requirements.txt']
for file in required_files:
    if not os.path.exists(file):
        raise FileNotFoundError(f"Required file {file} not found")
print("All required files found ✓")
agentcore_runtime = Runtime()

auth_config = {
    "customJWTAuthorizer": {
        "allowedClients": [
            runtime_client_id
        ],
        "discoveryUrl": runtime_cognito_discovery_url
    }
}

print("Configuring AgentCore Runtime...")
response = agentcore_runtime.configure(
    entrypoint="mcp_server_updated.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region,
    authorizer_configuration=auth_config,
    protocol="MCP",
    agent_name="ac_gateway_mcp_server"
)
print("Configuration completed ✓")

### Step 5.4: List tools through the gateway — still stale

Call `tools/list`. The new `cancelOrder` should NOT appear yet, because the gateway's catalog was last synced before we added the tool.

In [ ]:
import json

import requests
from strands.models import BedrockModel
from mcp.client.streamable_http import streamablehttp_client
from strands.tools.mcp.mcp_client import MCPClient
from strands import Agent


def get_token():
    token = utils.get_token(gw_user_pool_id, gw_client_id, gw_client_secret, scopeString, REGION)
    return token['access_token']


def create_streamable_http_transport():
    return streamablehttp_client(
        gatewayURL, headers={"Authorization": f"Bearer {get_token()}"}
    )


client = MCPClient(create_streamable_http_transport)

## The IAM group/user/ configured in ~/.aws/credentials should have access to Bedrock model
yourmodel = BedrockModel(
    model_id="global.anthropic.claude-haiku-4-5-20251001-v1:0", # may need to update model_id depending on region
    temperature=0.7,
    max_tokens=500,  # Limit response length
)

with client:
    # Call the listTools
    tools = client.list_tools_sync()
    # Create an Agent with the model and tools
    agent = Agent(
        model=yourmodel, tools=tools
    )  ## you can replace with any model you like
    # Invoke the agent with the sample prompt. This will only invoke MCP listTools and retrieve the list of tools the LLM has access to. The below does not actually call any tool.
    agent("Hi, can you list all tools available to you")  

### Step 5.5: Call `SynchronizeGatewayTargets`

In [ ]:
sync_response = gateway_client.synchronize_gateway_targets(
    gatewayIdentifier=gatewayID,
    targetIdList=[gatewayTargetID],
)
print(sync_response)

### Verificar as definições de tools atuais após a sincronização

Vamos invocar um agente strands para listar as MCP tools usando o Bedrock AgentCore Gateway. Isso retornará a nova lista de tools.

In [ ]:
sleep(10)
print(json.dumps(mcp.list_tools(), indent=2))

## Step 6: Implicit synchronization with `UpdateGatewayTarget`

### Step 6.1: Background

`CreateGatewayTarget` and `UpdateGatewayTarget` are also **control-plane operations on `listingMode='DEFAULT'` targets**, and they perform the same catalog refill as `SynchronizeGatewayTargets` — just bundled into the same call as the create/update. `CreateGatewayTarget` is the very first cache fill for a new target; `UpdateGatewayTarget` refills it as an automatic side effect of every update. No separate sync call is needed afterwards.

Like explicit sync, this only matters for DEFAULT-mode targets. DYNAMIC targets don't have a cache to fill.

Below we add `deleteOrder` to the MCP server, redeploy, then call `UpdateGatewayTarget` on the existing target. The catalog refresh happens implicitly.

![Diagram](images/mcp-server-target-implicit-sync.png)

### Step 6.2: Update the MCP server (add `deleteOrder`)

In [ ]:
%%writefile mcp_server_updated.py
from mcp.server.fastmcp import FastMCP

mcp = FastMCP(host="0.0.0.0", stateless_http=True)

@mcp.tool()
def getOrder() -> int:
    """Get an order"""
    return 123

@mcp.tool()
def updateOrder(orderId: int) -> int:
    """Update existing order"""
    return 456

@mcp.tool()
def cancelOrder(orderId: int) -> int:
    """cancel existing order"""
    return 789

@mcp.tool()
def deleteOrder(orderId: int) -> int:
    """delete existing order"""
    return 101
    
if __name__ == "__main__":
    mcp.run(transport="streamable-http")

Em seguida, vamos configurar o AgentCore Runtime para implantar o MCP server atualizado com a nova tool 'deleteOrder'.

In [ ]:
from bedrock_agentcore_starter_toolkit import Runtime
from boto3.session import Session

boto_session = Session()
region = boto_session.region_name
print(f"Using AWS region: {region}")

required_files = ['mcp_server_updated.py', 'requirements.txt']
for file in required_files:
    if not os.path.exists(file):
        raise FileNotFoundError(f"Required file {file} not found")
print("All required files found ✓")
agentcore_runtime = Runtime()

auth_config = {
    "customJWTAuthorizer": {
        "allowedClients": [
            runtime_client_id
        ],
        "discoveryUrl": runtime_cognito_discovery_url
    }
}

print("Configuring AgentCore Runtime...")
response = agentcore_runtime.configure(
    entrypoint="mcp_server_updated.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region,
    authorizer_configuration=auth_config,
    protocol="MCP",
    agent_name="ac_gateway_mcp_server"
)
print("Configuration completed ✓")

## Atualizar o Gateway Target existente com o MCP Server

In [ ]:
import boto3

gateway_client = boto3.client('bedrock-agentcore-control', region_name=REGION)
update_gateway_target_response = gateway_client.update_gateway_target(
    gatewayIdentifier=gatewayID,
    targetId = gatewayTargetID,
    name = 'mcp-server-target',
    targetConfiguration={
        'mcp': {
            'mcpServer': {
                'endpoint': agent_url
            }
        }
    },
    credentialProviderConfigurations=[
        {
            'credentialProviderType': 'OAUTH',
            'credentialProvider': {
                'oauthCredentialProvider': {
                    'providerArn': cognito_provider_arn,
                    'scopes': [
                        runtimeScopeString
                    ]
                }
            }
        },
    ]
)

print(update_gateway_target_response)

### Step 6.5: List tools again — the implicit sync caught the new tool

In [ ]:
sleep(10)
print(json.dumps(mcp.list_tools(), indent=2))

## Step 7: Dynamic listing with `listingMode='DYNAMIC'`

### Step 7.1: Background — DEFAULT vs DYNAMIC

By default, AgentCore Gateway *caches* the capabilities (tools, prompts, resources, resource templates) it discovered when the target was created, updated, or last synchronized. With `listingMode='DEFAULT'`, the four MCP list operations are answered from Gateway's catalog **without invoking the upstream MCP server**. Fast and resilient, but stale until the next sync.

With `listingMode='DYNAMIC'`, every list request is forwarded to the upstream MCP server, and no synchronization is ever required.

A few things to note:

- DYNAMIC mode is **not interoperable with semantic search** (`x_amz_bedrock_agentcore_search`) or with outbound three-legged OAuth (3LO).
- DYNAMIC mode applies uniformly across all four primitive types — tools, prompts, resources, and resource templates.

### Step 7.2: Extend the MCP server with prompts, resources, and a resource template

To demonstrate DEFAULT vs DYNAMIC for **all four** list operations, the upstream MCP server needs to expose all four primitive types. Rewrite `mcp_server_updated.py` to add prompts and resources alongside the existing tools, and add a fresh tool `archiveOrder` so the cached/live contrast is visible on the tools axis too.

In [ ]:
%%writefile mcp_server.py
import json

from mcp.server.fastmcp import FastMCP

mcp = FastMCP(host="0.0.0.0", stateless_http=True)


@mcp.tool()
def getOrder() -> int:
    """Get an order"""
    return 123


@mcp.tool()
def updateOrder(orderId: int) -> int:
    """Update existing order"""
    return 456


@mcp.tool()
def cancelOrder(orderId: int) -> int:
    """Cancel existing order"""
    return 789


@mcp.tool()
def deleteOrder(orderId: int) -> int:
    """Delete existing order"""
    return 101


@mcp.tool()
def archiveOrder(orderId: int) -> int:
    """Archive existing order"""
    return 202


if __name__ == "__main__":
    mcp.run(transport="streamable-http")

### Step 7.3: Re-deploy Runtime


In [ ]:
print("Re-launching the MCP server with the live additions...")
launch_result = agentcore_runtime.launch()
print(f"Agent ARN (unchanged): {launch_result.agent_arn}")

### Step 7.4: Create a new gateway target with `listingMode='DYNAMIC'`

Rather than mutate the existing `mcp-server-target` (which uses the default `listingMode='DEFAULT'`), create a *separate* target so both modes coexist on the same gateway and can be compared side-by-side. Both targets point at the same upstream MCP server URL, but they will report different capabilities depending on whether they read from a cache or from the live server.

In [ ]:
launch_result

In [ ]:
create_dynamic_target_response = gateway_client.create_gateway_target(
    name="mcp-server-target-dynamic",
    gatewayIdentifier=gatewayID,
    targetConfiguration={
        "mcp": {
            "mcpServer": {
                "endpoint": mcp_url,
                "listingMode": "DYNAMIC",
            }
        }
    },
    credentialProviderConfigurations=[
        {
            "credentialProviderType": "OAUTH",
            "credentialProvider": {
                "oauthCredentialProvider": {
                    "providerArn": cognito_provider_arn,
                    "scopes": [runtimeScopeString],
                }
            },
        },
    ],
)
dynamicTargetID = create_dynamic_target_response["targetId"]
print(f"Created DYNAMIC target: {dynamicTargetID}")

### Step 7.5: Side-by-side list tools (before any live changes)

Both targets currently point at the same MCP server URL. The DEFAULT target's catalog is whatever was last synced (from Steps 5 and 6 above). The DYNAMIC target was just created and will fetch its capabilities live on every list call.

> **Pagination is per-target, cached-first.** When multiple targets are attached, `tools/list` returns **one target's tools per page**, with a `nextCursor` for the next target. **DEFAULT-mode (cached) targets are paged first**, then DYNAMIC targets — so a single call without a cursor only surfaces the first DEFAULT target's tools. To see the merged catalog across both targets you have to follow `nextCursor` until it's empty. The `mcp.list_all_tools()` helper below does that loop for you (see `gateway_mcp_client.py`).

Expected:

- **First page (`mcp-server-target___`, DEFAULT):** the catalog from the last sync in Steps 5/6 — `getOrder`, `updateOrder`, `cancelOrder`, `deleteOrder`.
- **Second page (`mcp-server-target-dynamic___`, DYNAMIC):** the live MCP server right now — 5 tools including `archiveOrder`.

In [ ]:
mcp.list_tools()

In [ ]:
all_tools = mcp.list_all_tools()
print(f"{len(all_tools)} tools across both targets:")
for t in all_tools:
    print(f"  - {t['name']}")

### Step 7.6: DEFAULT vs DYNAMIC summary

| Aspect | DEFAULT | DYNAMIC |
|---|---|---|
| `tools/list`, `prompts/list`, `resources/list`, `resources/templates/list` | served from Gateway cache | forwarded to MCP server live |
| `tools/call`, `prompts/get`, `resources/read` | live to MCP server | live to MCP server |
| Requires `SynchronizeGatewayTargets` after capability changes | yes | no |
| Compatible with semantic search (`x_amz_bedrock_agentcore_search`) | yes | no |
| Compatible with outbound 3LO OAuth | yes | no |

# Limpeza
Recursos adicionais também são criados, como IAM role, IAM Policies, provedor de credenciais, funções AWS Lambda, Cognito user pools, buckets s3 que você pode precisar excluir manualmente como parte da limpeza. Isso depende do exemplo que você executar.

In [ ]:
import boto3
gateway_client = boto3.client('bedrock-agentcore-control', region_name=REGION)

utils.delete_gateway(gateway_client, gatewayID)